# Bonus — Beam Search Decoder

Membandingkan greedy decoding vs beam search (k=3, k=5) pada SimpleRNN decoder terbaik.

- Metrik: BLEU-4, METEOR, waktu inferensi
- Analisis kualitatif: contoh caption greedy vs beam search
- Kesimpulan pengaruh beam size

In [ ]:
import os, sys, json, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import nltk
from pathlib import Path


def _find_root(marker='requirements.txt'):
    p = Path(os.getcwd())
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError('Repo root not found')


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from shared.caption_utils import clean_caption, load_vocabulary
from shared.metrics import bleu4, meteor
from rnn.model import RNNDecoder
from rnn.train_keras import build_rnn_decoder
from bonus.beam_search import beam_search_rnn

print('Root:', REPO_ROOT)

## Config

In [ ]:
FEATURE_DIM  = 2048
EMBED_DIM    = 256
MAX_CAP_LEN  = 30

FEATURES_NPY  = 'features/flickr8k_features.npy'
VOCAB_JSON    = 'features/vocab.json'
IDX_JSON      = 'features/flickr8k_idx.json'
SPLITS_JSON   = 'features/splits.json'
CAPTIONS_TXT  = 'data/flickr8k/captions.txt'
IMAGES_DIR    = 'data/flickr8k/Images'
MODELS_DIR    = 'models/rnn'
BONUS_DIR     = 'models/beam_search'

os.makedirs(BONUS_DIR, exist_ok=True)

# Best model dari notebook 08
BEST_MODEL_NAME = 'rnn-1layer-512'
BEST_RNN_LAYERS = 1
BEST_RNN_UNITS  = 512

BEAM_SIZES = [3, 5]

## Load Resources

In [ ]:
features  = np.load(FEATURES_NPY)
vocab     = load_vocabulary(VOCAB_JSON)
idx2word  = {v: k for k, v in vocab.items()}
VOCAB_SIZE = len(vocab)

with open(IDX_JSON)    as f: idx_map = json.load(f)
with open(SPLITS_JSON) as f: splits  = json.load(f)

caption_map = {}
with open(CAPTIONS_TXT) as f:
    next(f)
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(',', 1)
        if len(parts) != 2:
            continue
        img_file, caption = parts
        img_id = os.path.splitext(img_file)[0]
        caption_map.setdefault(img_id, []).append(caption)

test_ids = splits['test']
test_feat_vecs = [features[idx_map[img_id]] for img_id in test_ids]

test_refs = [
    [clean_caption(c).split() for c in caption_map[img_id]]
    for img_id in test_ids
]

print(f'Vocab: {VOCAB_SIZE} | Features: {features.shape} | Test: {len(test_ids)}')

## Load Best Model

In [ ]:
keras_model = build_rnn_decoder(
    vocab_size=VOCAB_SIZE,
    feature_dim=FEATURE_DIM,
    embed_dim=EMBED_DIM,
    rnn_units=BEST_RNN_UNITS,
    num_rnn_layers=BEST_RNN_LAYERS,
)
keras_model.load_weights(os.path.join(MODELS_DIR, f'{BEST_MODEL_NAME}.keras'))
keras_model.summary()

decoder = RNNDecoder()
decoder.load_weights(keras_model, vocab)
print('\nDecoder loaded.')

## Greedy Decoding (Baseline)

In [ ]:
print('Running greedy decoding ...')
t0 = time.time()
greedy_caps = [
    decoder.generate_caption(fv, max_len=MAX_CAP_LEN)
    for fv in test_feat_vecs
]
greedy_time = time.time() - t0

greedy_b4  = bleu4(test_refs, [c.split() for c in greedy_caps])
greedy_met = meteor(test_refs, [c.split() for c in greedy_caps])

print(f'Greedy | BLEU-4: {greedy_b4:.4f}  METEOR: {greedy_met:.4f}  time: {greedy_time:.1f}s  '
      f'({greedy_time/len(test_ids)*1000:.1f} ms/img)')

## Beam Search (k=3 dan k=5)

In [ ]:
beam_results = {}

for k in BEAM_SIZES:
    print(f'Running beam search k={k} ...')
    t0 = time.time()
    beam_caps = [
        beam_search_rnn(
            decoder,
            fv,
            vocab,
            idx2word,
            beam_size=k,
            max_len=MAX_CAP_LEN,
        )
        for fv in test_feat_vecs
    ]
    elapsed = time.time() - t0

    b4  = bleu4(test_refs, [c.split() for c in beam_caps])
    met = meteor(test_refs, [c.split() for c in beam_caps])

    beam_results[k] = {
        'captions':   beam_caps,
        'bleu4':      b4,
        'meteor':     met,
        'time_total': elapsed,
        'time_per_img': elapsed / len(test_ids),
    }
    print(f'  Beam k={k} | BLEU-4: {b4:.4f}  METEOR: {met:.4f}  time: {elapsed:.1f}s  '
          f'({elapsed/len(test_ids)*1000:.1f} ms/img)')

## Tabel Perbandingan

In [ ]:
import pandas as pd

rows = [
    {
        'Method':       'Greedy',
        'Beam Size':    1,
        'BLEU-4':       round(greedy_b4,  4),
        'METEOR':       round(greedy_met, 4),
        'Total time (s)':    round(greedy_time, 1),
        'ms / image':   round(greedy_time / len(test_ids) * 1000, 1),
    }
]
for k in BEAM_SIZES:
    r = beam_results[k]
    rows.append({
        'Method':       f'Beam Search',
        'Beam Size':    k,
        'BLEU-4':       round(r['bleu4'],  4),
        'METEOR':       round(r['meteor'], 4),
        'Total time (s)':    round(r['time_total'], 1),
        'ms / image':   round(r['time_per_img'] * 1000, 1),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df

## Plot BLEU-4 & METEOR vs Beam Size

In [ ]:
all_k     = [1] + BEAM_SIZES
all_b4    = [greedy_b4]  + [beam_results[k]['bleu4']  for k in BEAM_SIZES]
all_met   = [greedy_met] + [beam_results[k]['meteor'] for k in BEAM_SIZES]
all_times = [greedy_time] + [beam_results[k]['time_total'] for k in BEAM_SIZES]
x_labels  = ['Greedy\n(k=1)'] + [f'Beam\n(k={k})' for k in BEAM_SIZES]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(x_labels, all_b4, color=['steelblue', 'darkorange', 'tomato'])
axes[0].set_ylabel('BLEU-4')
axes[0].set_title('BLEU-4 vs Decoding Strategy')
for i, v in enumerate(all_b4):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)

axes[1].bar(x_labels, all_met, color=['steelblue', 'darkorange', 'tomato'])
axes[1].set_ylabel('METEOR')
axes[1].set_title('METEOR vs Decoding Strategy')
for i, v in enumerate(all_met):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)

axes[2].bar(x_labels, all_times, color=['steelblue', 'darkorange', 'tomato'])
axes[2].set_ylabel('Waktu Total (s)')
axes[2].set_title('Waktu Inferensi Total')
for i, v in enumerate(all_times):
    axes[2].text(i, v + 0.5, f'{v:.1f}s', ha='center', fontsize=9)

plt.suptitle(f'Greedy vs Beam Search — {BEST_MODEL_NAME}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(BONUS_DIR, 'greedy_vs_beam.png'), dpi=120)
plt.show()

## Qualitative Analysis — Greedy vs Beam Search

Pilih 10 gambar dari test set (skor tinggi, sedang, rendah) dan bandingkan output greedy vs beam k=3 vs k=5.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4


def sentence_bleu4(refs, hyp):
    return sentence_bleu(refs, hyp, weights=(0.25,) * 4, smoothing_function=smoothie)


per_image_greedy = [
    sentence_bleu4(test_refs[i], greedy_caps[i].split())
    for i in range(len(test_ids))
]

sorted_idx = np.argsort(per_image_greedy)
n = len(sorted_idx)

low_idx  = list(sorted_idx[:4])
mid_idx  = list(sorted_idx[n // 2 - 2 : n // 2 + 2])
high_idx = list(sorted_idx[-4:][::-1])

selected = (high_idx + mid_idx + low_idx)[:10]
print('Selected:', selected)
print('Greedy BLEU-4:', [round(per_image_greedy[i], 4) for i in selected])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(24, 10))
axes = axes.flatten()

best_k = max(BEAM_SIZES, key=lambda k: beam_results[k]['bleu4'])

for plot_i, img_idx in enumerate(selected):
    img_id   = test_ids[img_idx]
    img_path = os.path.join(IMAGES_DIR, img_id + '.jpg')

    ax = axes[plot_i]
    if os.path.exists(img_path):
        ax.imshow(mpimg.imread(img_path))
    ax.axis('off')

    g_cap  = greedy_caps[img_idx]
    bk_cap = beam_results[best_k]['captions'][img_idx]
    gt_cap = caption_map[img_id][0]
    score  = per_image_greedy[img_idx]

    title = (
        f'[Greedy BLEU-4: {score:.3f}]\n'
        f'Greedy: {g_cap[:60]}\n'
        f'Beam k={best_k}: {bk_cap[:60]}\n'
        f'GT: {gt_cap[:60]}'
    )
    ax.set_title(title, fontsize=6.5, loc='left')

plt.suptitle(f'Qualitative: Greedy vs Beam k={best_k} — {BEST_MODEL_NAME}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(BONUS_DIR, 'qualitative_beam_search.png'), dpi=100)
plt.show()

## Perbandingan Caption untuk Setiap Strategi Decoding (Tabular)

In [ ]:
print(f'{"Idx":<5} {"Greedy":<50} {"Beam k=3":<50} {"Beam k=5":<50} {"Ground Truth"}')
print('-' * 210)
for img_idx in selected:
    img_id = test_ids[img_idx]
    g   = greedy_caps[img_idx][:48]
    b3  = beam_results[3]['captions'][img_idx][:48] if 3 in beam_results else '-'
    b5  = beam_results[5]['captions'][img_idx][:48] if 5 in beam_results else '-'
    gt  = caption_map[img_id][0][:48]
    print(f'{img_idx:<5} {g:<50} {b3:<50} {b5:<50} {gt}')

## Analisis Perbedaan Greedy vs Beam Search per Gambar

In [ ]:
per_image_beam = {}
for k in BEAM_SIZES:
    per_image_beam[k] = [
        sentence_bleu4(test_refs[i], beam_results[k]['captions'][i].split())
        for i in range(len(test_ids))
    ]

# Hitung berapa gambar yang naik/turun/sama dengan beam search
print(f'{'Method':<15} {'Better than greedy':<22} {'Worse':<10} {'Same':<10} {'Avg diff'}')
print('-' * 70)
for k in BEAM_SIZES:
    diffs  = np.array(per_image_beam[k]) - np.array(per_image_greedy)
    better = int((diffs > 1e-6).sum())
    worse  = int((diffs < -1e-6).sum())
    same   = len(diffs) - better - worse
    print(f'Beam k={k:<8} {better:<22} {worse:<10} {same:<10} {diffs.mean():.5f}')

In [ ]:
# Distribusi perbedaan BLEU-4 per gambar
fig, axes = plt.subplots(1, len(BEAM_SIZES), figsize=(12, 4))
if len(BEAM_SIZES) == 1:
    axes = [axes]

for ax, k in zip(axes, BEAM_SIZES):
    diffs = np.array(per_image_beam[k]) - np.array(per_image_greedy)
    ax.hist(diffs, bins=40, color='steelblue', edgecolor='white')
    ax.axvline(0, color='red', linestyle='--', label='no change')
    ax.set_xlabel('BLEU-4 diff (beam - greedy)')
    ax.set_ylabel('Jumlah gambar')
    ax.set_title(f'Distribusi perubahan BLEU-4 (k={k})')
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.suptitle('Per-image BLEU-4: Beam Search vs Greedy', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(BONUS_DIR, 'bleu4_diff_distribution.png'), dpi=120)
plt.show()

## Kesimpulan

Analisis perbandingan greedy vs beam search (k=3, k=5) pada SimpleRNN decoder:

In [ ]:
print('RINGKASAN — GREEDY VS BEAM SEARCH')
print(f'Model: {BEST_MODEL_NAME}')
print(f'Test set: {len(test_ids)} gambar\n')

all_methods = [('Greedy (k=1)', greedy_b4, greedy_met, greedy_time)]
for k in BEAM_SIZES:
    r = beam_results[k]
    all_methods.append((f'Beam (k={k})', r['bleu4'], r['meteor'], r['time_total']))

print(f'{'Method':<18} {'BLEU-4':>8} {'METEOR':>8} {'Time (s)':>10} {'Speedup vs greedy':>20}')
print('-' * 70)
base_time = greedy_time
for name, b4, met, t in all_methods:
    slowdown = t / base_time
    print(f'{name:<18} {b4:>8.4f} {met:>8.4f} {t:>10.1f} {slowdown:>18.2f}x')

print('\nKesimpulan:')
best_beam_k = max(BEAM_SIZES, key=lambda k: beam_results[k]['bleu4'])
best_beam_b4 = beam_results[best_beam_k]['bleu4']
delta = best_beam_b4 - greedy_b4
pct = delta / greedy_b4 * 100 if greedy_b4 > 0 else 0
slowdown = beam_results[best_beam_k]['time_total'] / greedy_time
print(f'- Beam search k={best_beam_k} menghasilkan BLEU-4 {best_beam_b4:.4f} '
      f'(delta={delta:+.4f}, {pct:+.1f}% dari greedy)')
print(f'- Tradeoff waktu: beam k={best_beam_k} adalah {slowdown:.1f}x lebih lambat dari greedy')
print(f'- Beam search efektif jika kenaikan BLEU-4 sebanding dengan overhead waktu')